[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/18_produccion/18_proyecto_produccion.ipynb)

# Proyecto 18 · Servicio Políglota: caché, versionado y plan de migración medido

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | ≈ 2 h |
| **GPU** | No necesaria (CPU, `FAST_DEV_RUN = True`, ≈ 1–2 min de ejecución) |
| **Requiere** | Lección 18 (caché, migración, filtros, costos) · módulos 13 (RRF), 14 (FAISS), 15 (bootstrap pareado) |
| **Conecta con** | Módulo 17 (privacidad de los vectores que sirves) · Módulo 19 (decisiones de diseño) · **Módulo 20** (este servicio es la capa de servicio del capstone) |

## 🏢 Contexto de negocio
Eres ML engineer en **Políglota Labs**. Mañana se abre la beta: el buscador debe funcionar como **servicio** y no como notebook.
La CTO pide tres cosas: *«(1) una API con caché y filtro por idioma que no se rompa cuando cambiemos de modelo; (2) números de latencia;
(3) un plan para migrar de v1 a v2 con costos y calidad medidos: ¿re-embebemos todo, usamos un adaptador o esperamos?»*

## 📦 Dataset
Corpus Políglota canónico (`carga_corpus` del módulo 00: Wikipedia ES/EN + CodeSearchNet; sin red, **corpus sintético** + funciones reales de la stdlib, con aviso).
Embedders v1/v2: `multilingual-e5-small` / `multilingual-e5-base` si hay hub; si no, los fallbacks TF-IDF+SVD de la lección (espacios incompatibles).
Los umbrales numéricos están **calibrados con la solución de referencia** en `FAST_DEV_RUN=True`, corpus sintético y fallbacks (semilla 42). Con modelos o datos reales el notebook imprime tus cifras y aplica solo los chequeos estructurales.

## 🎯 Entregables y rúbrica (100 puntos)
| # | Entregable | Criterio objetivo (calibrado con la solución) | Puntos |
|---|---|---|---|
| 1 | Caché LRU + clave con versión y normalización | `test_cache` pasa (orden LRU, contadores, NFC, versión en la clave) | 15 |
| 2 | `ServicioPoliglota.buscar` (denso + BM25 + RRF + filtro) | filtro respetado; nDCG@10 híbrido ≥ 0.844 (ref. 0.864) | 25 |
| 3 | Latencia y caché | p50 en caliente ≤ p50 en frío; hit rate con tráfico Zipf ≥ 0.755 (ref. 0.805) | 15 |
| 4 | Adaptador v2→v1 servido como versión | nDCG@10 con 50 % de pares ≥ 0.787 (ref. 0.817) y < nDCG de v2 completo | 20 |
| 5 | Plan de migración | dict con costos (API y GPU, supuestos explícitos), 2 tests pareados y una decisión coherente con la regla | 20 |
| 6 | Reflexión | 3 respuestas concretas al final | 5 |

## 0. Preparación (celdas dadas)

In [ ]:
!pip install -q faiss-cpu qdrant-client fastapi httpx sentence-transformers scikit-learn matplotlib pandas

In [ ]:
import os, sys, time, math, random, json, hashlib, unicodedata, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True       # True: tamaños diminutos (CPU en minutos). False en Colab con GPU para modelos reales y más datos.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
try:
    import torch
    torch.manual_seed(SEED); torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, device = None, "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """IC percentil de la media remuestreando QUERIES (la unidad de muestreo es la query, no el par query-doc)."""
    v = np.asarray(values, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def paired_bootstrap(a, b, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42) -> dict:
    """Test pareado A vs B sobre las MISMAS queries. IC percentil de la diferencia media y p-valor bilateral
    por 'shift' (se centra la diferencia en 0 bajo H0 y se cuenta cuántas veces el remuestreo supera lo observado)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    null = (d - d.mean())[idx].mean(axis=1)
    p = (np.sum(np.abs(null) >= abs(d.mean())) + 1) / (n_boot + 1)
    return {"diff": float(d.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "p_value": float(p)}

def paired_permutation(a, b, n_perm: int = 10000, seed: int = 42) -> float:
    """Test de aleatorización pareado (sign-flip): bajo H0 el signo de cada diferencia por query es intercambiable."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(d)))
    stat = np.abs((signs * d).mean(axis=1))
    return float((np.sum(stat >= abs(d.mean()) - 1e-12) + 1) / (n_perm + 1))

def holm(pvalues: dict, alpha: float = 0.05) -> dict:
    """Corrección de Holm-Bonferroni para comparaciones múltiples. Devuelve {nombre: (p_ajustado, rechaza_H0)}."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for i, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[name] = (running, running < alpha)
    return out

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]; doc_texts = [d["text"] for d in docs]
doc_lang = np.array([d["lang"] for d in docs])
qids = [q["qid"] for q in queries]; q_texts = [q["text"] for q in queries]; q_lang = np.array([q["lang"] for q in queries])
print(f"fuente={corpus['source']} · {len(docs)} docs · {len(queries)} queries · idiomas={sorted(set(doc_lang))}")

In [ ]:
def run_desde_scores(S, k: int = 10, ids=None) -> dict:
    """Matriz de scores (n_queries x n_docs) -> run {qid: [doc_ids top-k]} con el desempate canónico."""
    ids = doc_ids if ids is None else ids
    return {qid: rank_from_scores(S[i], ids, k) for i, qid in enumerate(qids)}

def ndcg_por_query(run: dict, k: int = 10) -> np.ndarray:
    """nDCG@k de cada query (orden de `qids`); unidad de remuestreo para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), qrels[q], k) for q in qids])

def l2n(X):
    X = np.asarray(X, dtype=np.float32)
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)

BM25_IDX = BM25().fit([tokenize_simple(t) for t in doc_texts], doc_ids)
S_BM25 = np.array([BM25_IDX.scores(tokenize_simple(q)) for q in q_texts])
print("BM25 (baseline del módulo 00):", {m: round(v, 3) for m, v in evaluate(run_desde_scores(S_BM25), qrels).items() if m in ("ndcg", "recall", "mrr")})

In [ ]:
# Dos versiones del embedder de Políglota: v1 (la que está en producción) y v2 (la candidata).
# Con hub: v1 = multilingual-e5-small (384 d), v2 = multilingual-e5-base (768 d).
# Sin hub: dos fallbacks TF-IDF+SVD DISTINTOS (otro analizador y otra dimensión) -> espacios incompatibles, como en la vida real.
USAR_MODELOS_REALES = True
V1_REAL, V2_REAL = "multilingual-e5-small", "multilingual-e5-base"

def carga_versiones(textos_ajuste) -> dict:
    vers = {}
    if USAR_MODELOS_REALES:
        for nombre, clave in (("v1", V1_REAL), ("v2", V2_REAL)):
            e = cargar_embedder(clave)          # None si no hay hub
            if e is None:
                break
            vers[nombre] = e
    if len(vers) < 2:
        print("⚠️  Sin modelos reales: v1/v2 = fallbacks TF-IDF+SVD (NO son modelos neuronales; las cifras no representan a e5).")
        vers = {"v1": Embedder("v1 · char 3-4 + SVD 48d (fallback)", {}, fallback=HashSVDEmbedder(dim=48, ngram=(3, 4)).fit(textos_ajuste)),
                "v2": Embedder("v2 · char 2-5 + SVD 128d (fallback)", {}, fallback=HashSVDEmbedder(dim=128, ngram=(2, 5)).fit(textos_ajuste))}
    return vers

VERS = carga_versiones(doc_texts)
EMB = {}
for v, e in VERS.items():
    t0 = time.perf_counter()
    D, Qv = e.encode_docs(doc_texts), e.encode_queries(q_texts)
    EMB[v] = {"D": D, "Q": Qv, "seg": time.perf_counter() - t0}
    print(f"{v}: {e.clave} · dim={D.shape[1]} · nDCG@10={evaluate(run_desde_scores(Qv @ D.T), qrels)['ndcg']:.3f} · {EMB[v]['seg']:.1f}s")

In [ ]:
def corre(f):
    """Ejecuta un chequeo sin romper 'ejecutar todo' si el TODO está pendiente."""
    try:
        r = f()
        print(r if r is not None else "✅ OK")
        return True
    except NotImplementedError as e:
        print(f"⏳ pendiente: {e}")
    except AssertionError as e:
        print(f"❌ falló: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"❌ {type(e).__name__}: {e}")
    return False

REF = {'hib_filtro': 0.882, 'hib': 0.864, 'hit_zipf': 0.805, 'adapt50': 0.817, 'v1': 0.786, 'v2': 0.861}
RES = {}   # aquí se guardan tus resultados para la rúbrica

## 1. Caché de embeddings de query (TODO 1)
Implementa la normalización, la clave (con **versión**) y una LRU con `OrderedDict`. Repasa §4 de la lección.

<details><summary>🪜 Pista 1 (la idea)</summary>

La clave debe cambiar si cambia la versión y **no** debe cambiar con mayúsculas/espacios/forma Unicode. La LRU es un `OrderedDict` donde el final es «lo más reciente».
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`unicodedata.normalize('NFC', t).lower().split()` + `' '.join(...)`; `hashlib.sha1(f'{version}\x00{norm}'.encode()).hexdigest()`; `self.d.move_to_end(k)` y `self.d.popitem(last=False)`.
</details>

In [ ]:
from collections import OrderedDict

def normaliza_query(texto: str) -> str:
    """NFC + minúsculas + espacios colapsados."""
    # TODO 1a
    raise NotImplementedError("TODO 1a: normaliza_query")

def clave_cache(version: str, texto: str) -> str:
    """sha1 de 'version + separador + texto normalizado' (hexdigest)."""
    # TODO 1b
    raise NotImplementedError("TODO 1b: clave_cache")

class LRUCache:
    """Caché LRU con contadores hits/misses y propiedad hit_rate."""
    def __init__(self, capacidad: int):
        self.cap, self.d, self.hits, self.misses = capacidad, OrderedDict(), 0, 0
    def get(self, k):
        # TODO 1c: si está, muévela al final (más reciente), suma un hit y devuélvela; si no, suma un miss y devuelve None
        raise NotImplementedError("TODO 1c: LRUCache.get")
    def put(self, k, v):
        # TODO 1d: inserta/actualiza como más reciente; si se supera la capacidad, expulsa la MENOS reciente
        raise NotImplementedError("TODO 1d: LRUCache.put")
    @property
    def hit_rate(self) -> float:
        n = self.hits + self.misses
        return self.hits / n if n else 0.0

In [ ]:
def test_cache():
    assert normaliza_query("  Cómo   INVALIDAR\tla caché ") == "cómo invalidar la caché"
    assert normaliza_query("café") == normaliza_query("café"), "NFC: 'e' + tilde combinante == 'é'"
    assert clave_cache("v1", "Hola  Mundo") == clave_cache("v1", "hola mundo"), "la normalización va antes del hash"
    assert clave_cache("v1", "hola") != clave_cache("v2", "hola"), "¡la versión debe ir en la clave!"
    c = LRUCache(2); c.put("a", 1); c.put("b", 2); assert c.get("a") == 1
    c.put("c", 3)                                   # expulsa 'b' (la menos reciente: 'a' se acaba de usar)
    assert c.get("b") is None and c.get("a") == 1 and c.get("c") == 3, "orden LRU incorrecto"
    assert (c.hits, c.misses) == (3, 1) and abs(c.hit_rate - 0.75) < 1e-12
    return "✅ test_cache OK"
corre(test_cache)

## 2. El servicio (TODO 2)
Te damos el esqueleto: índice FAISS por versión, BM25 compartido, alias `prod`, generación de candidatos con filtro **dentro** de la búsqueda (`IDSelectorBatch`) y métricas. Te faltan `embebe_query` (con caché y adaptador opcional) y `buscar` (fusión RRF).

<details><summary>🪜 Pista 1 (la idea)</summary>

`buscar` no calcula scores densos ni BM25 a mano: llama a `_rangos_densos` / `_rangos_bm25`, que ya devuelven índices de documento ordenados. Tu trabajo es fusionarlos con RRF y medir el tiempo.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`listas = []`; `if modo in ('denso','hibrido'): listas.append(self._rangos_densos(texto, nombre, cand))`; `for lst in listas: for r, i in enumerate(lst): score[i] = score.get(i, 0) + 1/(self.rrf_k + r + 1)`; `top = sorted(score, key=lambda i: (-score[i], i))[:k]`.
</details>

In [ ]:
import faiss
faiss.omp_set_num_threads(1)

class ServicioPoliglota:
    """Búsqueda versionada: caché de embeddings de query + índice denso por versión (FAISS IndexFlatIP) + BM25,
    filtro por idioma, fusión RRF, alias 'prod' -> versión y versiones 'adaptador' (queries nuevas -> índice viejo)."""
    def __init__(self, docs, bm25, cache_cap: int = 2048, rrf_k: int = 60, profundidad: int = 50, tokenizador=None,
                 fusion: str = "rrf", alfa: float = 0.5):
        # CONTRATO: `tokenizador` debe ser el MISMO con el que se construyó `bm25` (si no, fallo silencioso F3 del módulo 19)
        self.tok = tokenizador or tokenize_simple
        self.fusion, self.alfa = fusion, alfa      # fusion: 'rrf' | 'lineal' (alfa·denso + (1-alfa)·BM25, min-max en cada lista)
        self.ids = [d["id"] for d in docs]; self.lang = np.array([d["lang"] for d in docs])
        self.bm25, self.cache, self.rrf_k, self.prof = bm25, LRUCache(cache_cap), rrf_k, profundidad
        self.versiones, self.alias, self.lat_ms = {}, {}, []

    def registrar_version(self, nombre, embedder, D, indice: str = "flat", ef_search: int = 64):
        """indice: 'flat' (exacto) | 'hnsw' (FAISS IndexHNSWFlat, M=32)."""
        if indice == "hnsw":
            idx = faiss.IndexHNSWFlat(D.shape[1], 32, faiss.METRIC_INNER_PRODUCT); idx.hnsw.efConstruction = 80
        else:
            idx = faiss.IndexFlatIP(D.shape[1])
        idx.add(l2n(D))
        self.versiones[nombre] = {"emb": embedder, "idx": idx, "W": None, "base": nombre, "indice": indice, "ef": ef_search}

    def registrar_adaptador(self, nombre, embedder, W, sobre: str):
        """Queries con `embedder` (versión nueva) proyectadas con W al espacio del índice `sobre` (versión vieja)."""
        base = self.versiones[sobre]
        self.versiones[nombre] = {"emb": embedder, "idx": base["idx"], "W": W, "base": sobre, "indice": base["indice"], "ef": base["ef"]}

    def embebe_query(self, texto, nombre):
        """Devuelve el vector (1, d) de la query para la versión `nombre`, usando la caché.
        Si la versión tiene adaptador W, aplica v @ W y renormaliza."""
        # TODO 2a: clave = clave_cache(nombre, texto); si está en self.cache, devuélvela; si no, calcula con
        #          self.versiones[nombre]["emb"].encode_queries([texto]), aplica W si existe (l2n(v @ W)) y guárdala.
        raise NotImplementedError("TODO 2a: embebe_query")

    def _rangos_densos(self, texto, nombre, cand):
        return [int(i) for i in self._cands_densos(texto, nombre, cand)[0]]

    def _cands_densos(self, texto, nombre, cand):
        """(índices, scores) del top-`profundidad` denso, con el filtro aplicado DENTRO de la búsqueda."""
        cfg = self.versiones[nombre]; idx = cfg["idx"]; q = self.embebe_query(texto, nombre)
        sel = faiss.IDSelectorBatch(cand.astype("int64")) if cand is not None else None
        if cfg["indice"] == "hnsw":       # HNSW exige sus propios parámetros (efSearch + selector de filtro)
            params = faiss.SearchParametersHNSW(efSearch=cfg["ef"]) if sel is None else faiss.SearchParametersHNSW(sel=sel, efSearch=cfg["ef"])
        else:
            params = faiss.SearchParameters(sel=sel) if sel is not None else None
        Dsc, I = idx.search(q, min(self.prof, idx.ntotal), params=params)
        m = I[0] >= 0
        return I[0][m], Dsc[0][m]

    def _lineal(self, texto, nombre, cand):
        """Combinación convexa con min-max dentro de cada lista (un doc ausente de una lista aporta 0 por ella)."""
        I, sd = self._cands_densos(texto, nombre, cand)
        sb_all = self.bm25.scores(self.tok(texto))
        Ib = np.array(self._rangos_bm25(texto, cand), dtype=int); sb = sb_all[Ib] if len(Ib) else np.zeros(0)
        def mm(v):
            return (v - v.min()) / max(float(v.max() - v.min()), 1e-9) if len(v) else v
        score = {}
        for i, v in zip(I, mm(sd)):
            score[int(i)] = score.get(int(i), 0.0) + self.alfa * float(v)
        for i, v in zip(Ib, mm(sb)):
            score[int(i)] = score.get(int(i), 0.0) + (1 - self.alfa) * float(v)
        return score

    def _rangos_bm25(self, texto, cand):
        s = self.bm25.scores(self.tok(texto))
        if cand is not None:
            m = np.full(len(s), -np.inf); m[cand] = s[cand]; s = m
        orden = np.lexsort((np.arange(len(s)), -s))[: self.prof]
        return [int(i) for i in orden if np.isfinite(s[i]) and s[i] > 0]

    def buscar(self, texto, k: int = 10, lang=None, modo: str = "hibrido", version: str = "prod"):
        """Devuelve [(doc_id, score_rrf)] top-k. modo: 'denso' | 'bm25' | 'hibrido'. lang: filtro DURO por idioma.
        Registra la latencia en self.lat_ms (ms)."""
        # TODO 2b: 1) resuelve el alias (self.alias.get(version, version)); 2) cand = índices con self.lang == lang (o None);
        #          3) listas de rangos con self._rangos_densos / self._rangos_bm25 según el modo;
        #          4) RRF: score[i] += 1/(self.rrf_k + rango + 1); 5) top-k por (-score, i); 6) guarda la latencia.
        raise NotImplementedError("TODO 2b: buscar")

    def metricas(self) -> dict:
        lat = np.array(self.lat_ms) if self.lat_ms else np.zeros(1)
        return {"n": len(self.lat_ms), "p50_ms": float(np.percentile(lat, 50)), "p99_ms": float(np.percentile(lat, 99)),
                "hit_rate": self.cache.hit_rate, "versiones": sorted(self.versiones), "alias": dict(self.alias)}

In [ ]:
def construye_servicio(cache_cap=2048):
    s = ServicioPoliglota(docs, BM25_IDX, cache_cap=cache_cap)
    for v in ("v1", "v2"):
        s.registrar_version(v, VERS[v], EMB[v]["D"])
    s.alias["prod"] = "v2"
    return s

def ndcg_servicio(s, **kw) -> np.ndarray:
    """nDCG@10 por query sirviendo TODAS las queries a través del servicio (como en producción)."""
    run = {qid: [d for d, _ in s.buscar(t, k=10, **kw)] for qid, t in zip(qids, q_texts)}
    return ndcg_por_query(run)

def test_servicio():
    s = construye_servicio()
    r = s.buscar(q_texts[0], k=10, lang="code")
    assert len(r) == 10 and all(doc_lang[doc_ids.index(d)] == "code" for d, _ in r), "el filtro de idioma no se respeta"
    assert all(r[i][1] >= r[i + 1][1] for i in range(len(r) - 1)), "resultados no ordenados por score"
    hib = ndcg_servicio(s).mean()
    s2 = construye_servicio()
    run_f = {qid: [d for d, _ in s2.buscar(t, k=10, lang=str(l))] for qid, t, l in zip(qids, q_texts, q_lang)}
    hib_f = ndcg_por_query(run_f).mean()
    RES.update(hib=hib, hib_filtro=hib_f)
    assert hib >= REF["hib"] - 0.02, f"nDCG híbrido {hib:.3f} < {REF['hib'] - 0.02:.3f}"
    assert s.cache.misses == len(set(clave_cache('v2', t) for t in q_texts)), "cada query distinta debe ser un miss la 1ª vez"
    return f"✅ test_servicio OK · híbrido={hib:.3f} · híbrido+filtro idioma={hib_f:.3f}"

corre(test_servicio)

## 3. Latencia y caché bajo tráfico (TODO 3)
Mide p50/p99 con batch = 1 en **frío** (caché vacía) y en **caliente** (las mismas queries otra vez), y el hit rate con tráfico Zipf (`trafico_zipf`, dado).

In [ ]:
def trafico_zipf(n, universo, s=1.1, p_variante=0.35, seed=SEED):
    """(Dado) n queries Zipf sobre un universo de intenciones, con variantes de mayúsculas/espacios."""
    rng = np.random.default_rng(seed)
    p = 1.0 / np.arange(1, len(universo) + 1) ** s; p /= p.sum()
    out = []
    for i in rng.choice(len(universo), size=n, p=p):
        t = universo[i]
        if rng.random() < p_variante:
            t = rng.choice([t.upper(), t.capitalize(), "  " + t, t.replace(" ", "  "), t + " "])
        out.append(t)
    return out
UNIVERSO = [q_texts[i % len(q_texts)] + ("" if i < len(q_texts) else f" {i // len(q_texts)}") for i in range(3000)]

<details><summary>🪜 Pista 1 (la idea)</summary>

Frío: servicio nuevo, recorre `q_texts` una vez. Caliente: recorre `q_texts` otra vez **en el mismo servicio**. `s.lat_ms` guarda las latencias en orden: sepáralas por tramos.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`s = construye_servicio(); for t in q_texts: s.buscar(t)`; `frio = np.array(s.lat_ms)`; repite y toma `np.array(s.lat_ms[len(q_texts):])`. Hit rate: servicio con `cache_cap=1024`, recorre `trafico_zipf(5000, UNIVERSO)` y lee `s.cache.hit_rate`.
</details>

In [ ]:
def mide_latencias() -> dict:
    """Devuelve {"frio_p50","frio_p99","caliente_p50","caliente_p99","hit_rate_zipf"} (ms y fracción)."""
    # TODO 3
    raise NotImplementedError("TODO 3: mide_latencias")

def test_latencias():
    m = mide_latencias(); RES.update(m)
    assert m["caliente_p50"] <= m["frio_p50"], "la caché no está acelerando nada: ¿guardas y lees con la misma clave?"
    assert m["hit_rate_zipf"] >= REF["hit_zipf"] - 0.05, f"hit rate {m['hit_rate_zipf']:.3f} bajo: ¿normalizas antes de hashear?"
    return "✅ test_latencias OK · " + " · ".join(f"{k}={v:.3f}" for k, v in m.items())
corre(test_latencias)

## 4. Adaptador v2 → v1 servido como versión (TODO 4)
Ajusta $W$ con el 50 % de los documentos embebidos con ambas versiones (ridge, $\lambda=0.1$), regístralo en el servicio como versión `v2→v1` **sobre el índice v1** y mide nDCG@10 sirviendo en modo denso.

<details><summary>🪜 Pista 1 (la idea)</summary>

Es la fórmula cerrada de ridge de la lección: $W=(A^\top A+\lambda I)^{-1}A^\top B$ con $A$ = embeddings v2 y $B$ = embeddings v1 de los **mismos** textos.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ B)`; Procrustes: `U, _, Vt = np.linalg.svd(A.T @ B, full_matrices=False); W = U @ Vt`. Luego `s.registrar_adaptador('v2→v1', VERS['v2'], W, sobre='v1')` y `ndcg_servicio(s, modo='denso', version='v2→v1')`.
</details>

In [ ]:
def ajusta_adaptador(A, B, lam: float = 0.1, metodo: str = "ridge"):
    """W tal que A @ W ≈ B. metodo: 'ridge' | 'procrustes'."""
    # TODO 4a
    raise NotImplementedError("TODO 4a: ajusta_adaptador")

def evalua_adaptador(frac: float = 0.5, seed: int = SEED) -> dict:
    """Devuelve {"v1": nDCG, "v2": nDCG, "adaptador": nDCG, "pq": {...por query...}} sirviendo por el servicio en modo denso."""
    # TODO 4b: muestra frac de documentos con np.random.default_rng(seed).choice(..., replace=False), ajusta W,
    #          registra 'v2→v1' y evalúa las tres versiones con ndcg_servicio(s, modo="denso", version=...)
    raise NotImplementedError("TODO 4b: evalua_adaptador")

def test_adaptador():
    r = evalua_adaptador(0.5); RES.update(adapt50=r["adaptador"], v1=r["v1"], v2=r["v2"], _pq=r["pq"])
    A = np.random.default_rng(0).normal(size=(50, 6)); Rm = np.linalg.qr(np.random.default_rng(1).normal(size=(6, 6)))[0]
    assert np.allclose(ajusta_adaptador(A, A @ Rm, metodo="procrustes"), Rm, atol=1e-6), "Procrustes debe recuperar una rotación exacta"
    assert r["adaptador"] >= REF["adapt50"] - 0.03, f"adaptador {r['adaptador']:.3f} demasiado bajo"
    assert r["adaptador"] < r["v2"], "un adaptador lineal no debería superar al re-embedding completo"
    return f"✅ test_adaptador OK · v1={r['v1']:.3f} · adaptador(50 %)={r['adaptador']:.3f} · v2={r['v2']:.3f}"
corre(test_adaptador)

## 5. Plan de migración con números (TODO 5)
Devuelve un `dict` con: `tokens_corpus_prod`, `coste_api_usd` (por proveedor), `coste_gpu_usd`, `horas_gpu`, `supuestos` (texto), `v2_vs_v1` y `v2_vs_adaptador` (salidas de `paired_bootstrap` sobre nDCG por query) y `decision`.

**Regla de decisión** (la que pide la CTO):
* si el IC95 de `v2 − v1` **no** excluye el 0 → `"quedarse en v1"`;
* si lo excluye y el IC95 de `v2 − adaptador` también → `"re-embedding blue/green (adaptador solo como puente)"`;
* si `v2 − v1` lo excluye pero `v2 − adaptador` no → `"adaptador"`.

Escenario de producción: 10 M de chunks de 400 tokens. Precios (oct-2026, verifícalos): 0.02, 0.13 y 0.15 $/1M; GPU **supuesta** 0.80 $/h, 20 000 tok/s, 60 % de utilización.

<details><summary>🪜 Pista 1 (la idea)</summary>

Reutiliza `RES['_pq']` (nDCG por query de v1, v2 y adaptador que guardó el TODO 4) y `paired_bootstrap(a, b)` del módulo 15.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`d = paired_bootstrap(pq['v2'], pq['v1'])`; «excluye el 0» ⇔ `d['ci_low'] > 0`. Costos: `T = 10_000_000 * 400`; `T / 1e6 * precio` y `T / (20_000 * 3600 * 0.6) * 0.80`.
</details>

In [ ]:
PRECIOS_API = {"OpenAI text-embedding-3-small": 0.02, "OpenAI text-embedding-3-large": 0.13, "Google gemini-embedding-001": 0.15}

def plan_migracion(pq: dict) -> dict:
    # TODO 5
    raise NotImplementedError("TODO 5: plan_migracion")

def test_plan():
    if "_pq" not in RES:
        raise NotImplementedError("completa antes el TODO 4 (guarda los nDCG por query)")
    p = plan_migracion(RES["_pq"]); RES["plan"] = p
    for k in ("tokens_corpus_prod", "coste_api_usd", "coste_gpu_usd", "horas_gpu", "supuestos", "v2_vs_v1", "v2_vs_adaptador", "decision"):
        assert k in p, f"falta la clave {k}"
    assert abs(p["coste_api_usd"]["OpenAI text-embedding-3-small"] - 80.0) < 1e-6 and abs(p["coste_gpu_usd"] - 74.07) < 0.01
    a, b = p["v2_vs_v1"]["ci_low"] > 0, p["v2_vs_adaptador"]["ci_low"] > 0
    esperada = "quedarse en v1" if not a else ("re-embedding blue/green (adaptador solo como puente)" if b else "adaptador")
    assert p["decision"] == esperada, f"decisión incoherente con la regla: {p['decision']!r} vs {esperada!r}"
    return f"✅ test_plan OK · decisión: {p['decision']}"
corre(test_plan)

## 6. (Opcional) El servicio detrás de HTTP
Si `fastapi` y `httpx` están instalados, esta celda expone tu servicio con FastAPI y lo prueba en proceso con `TestClient`. En Colab puedes lanzarlo con `uvicorn` en segundo plano.

In [ ]:
# (Opcional) El mismo servicio detrás de una API HTTP con FastAPI. En Colab puedes servirlo con uvicorn + túnel;
# aquí lo probamos en proceso con TestClient (requiere httpx). Si falta alguna librería, la celda solo avisa.
try:
    from fastapi import FastAPI
    from fastapi.testclient import TestClient
    from pydantic import BaseModel

    class Peticion(BaseModel):
        q: str
        k: int = 10
        lang: str | None = None
        modo: str = "hibrido"
        version: str = "prod"

    def crea_app(servicio) -> "FastAPI":
        app = FastAPI(title="Políglota Search")
        @app.get("/health")
        def health():
            return {"ok": True, "prod": servicio.alias.get("prod")}
        @app.post("/search")
        def search(p: Peticion):
            res = servicio.buscar(p.q, k=p.k, lang=p.lang, modo=p.modo, version=p.version)
            return {"version": servicio.alias.get(p.version, p.version), "results": [{"id": i, "score": s} for i, s in res]}
        @app.get("/metrics")
        def metrics():
            return servicio.metricas()
        return app
    FASTAPI_OK = True
except Exception as e:  # noqa: BLE001
    FASTAPI_OK = False
    print(f"⚠️ FastAPI/httpx no disponibles ({type(e).__name__}); celda opcional omitida.")

In [ ]:
def prueba_api():
    if not FASTAPI_OK:
        return "omitido"
    cli = TestClient(crea_app(construye_servicio()))
    assert cli.get("/health").json()["prod"] == "v2"
    r = cli.post("/search", json={"q": q_texts[0], "k": 5, "lang": "es"}).json()
    assert r["version"] == "v2" and len(r["results"]) == 5
    return f"✅ API OK · /search -> {r['results'][:2]} · /metrics -> {cli.get('/metrics').json()['n']} peticiones"
corre(prueba_api)

In [ ]:
def rubrica():
    filas = [("1 caché", 15, corre(test_cache)), ("2 servicio", 25, corre(test_servicio)), ("3 latencia", 15, corre(test_latencias)),
             ("4 adaptador", 20, corre(test_adaptador)), ("5 plan", 20, corre(test_plan))]
    total = sum(p for _, p, ok in filas if ok)
    print("\n".join(f"{'✅' if ok else '⬜'} {n:12s} {p:3d} pts" for n, p, ok in filas))
    print(f"Puntuación automática: {total}/95 (+5 de la reflexión, la revisa una persona)")
    if corpus["source"] != "synthetic":
        print("ℹ️ Datos reales: los umbrales numéricos están calibrados con el corpus sintético; compara tus cifras con criterio.")
rubrica()

---
## ⛔ SPOILER — intenta resolverlo primero
A partir de aquí está la **solución de referencia** completa. Ejecutarla redefine tus funciones y vuelve a pasar la rúbrica.

In [ ]:
from collections import OrderedDict

def normaliza_query(texto: str) -> str:
    """Normalización ANTES de la clave de caché: NFC + minúsculas + espacios colapsados (sube el hit rate)."""
    return " ".join(unicodedata.normalize("NFC", texto).lower().split())

def clave_cache(version: str, texto: str) -> str:
    """La versión del modelo VA EN LA CLAVE: si no, tras migrar servirías vectores v1 contra un índice v2."""
    return hashlib.sha1(f"{version}\x00{normaliza_query(texto)}".encode("utf-8")).hexdigest()

class LRUCache:
    """Caché LRU O(1) con OrderedDict y contadores de aciertos/fallos."""
    def __init__(self, capacidad: int):
        self.cap, self.d, self.hits, self.misses = capacidad, OrderedDict(), 0, 0
    def get(self, k):
        if k in self.d:
            self.d.move_to_end(k); self.hits += 1
            return self.d[k]
        self.misses += 1
        return None
    def put(self, k, v):
        self.d[k] = v; self.d.move_to_end(k)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)          # expulsa el menos usado recientemente
    @property
    def hit_rate(self) -> float:
        n = self.hits + self.misses
        return self.hits / n if n else 0.0

In [ ]:
import faiss
faiss.omp_set_num_threads(1)

class ServicioPoliglota:
    """Búsqueda versionada: caché de embeddings de query + índice denso por versión (FAISS IndexFlatIP) + BM25,
    filtro por idioma, fusión RRF, alias 'prod' -> versión y versiones 'adaptador' (queries nuevas -> índice viejo)."""
    def __init__(self, docs, bm25, cache_cap: int = 2048, rrf_k: int = 60, profundidad: int = 50, tokenizador=None,
                 fusion: str = "rrf", alfa: float = 0.5):
        # CONTRATO: `tokenizador` debe ser el MISMO con el que se construyó `bm25` (si no, fallo silencioso F3 del módulo 19)
        self.tok = tokenizador or tokenize_simple
        self.fusion, self.alfa = fusion, alfa      # fusion: 'rrf' | 'lineal' (alfa·denso + (1-alfa)·BM25, min-max en cada lista)
        self.ids = [d["id"] for d in docs]; self.lang = np.array([d["lang"] for d in docs])
        self.bm25, self.cache, self.rrf_k, self.prof = bm25, LRUCache(cache_cap), rrf_k, profundidad
        self.versiones, self.alias, self.lat_ms = {}, {}, []

    def registrar_version(self, nombre, embedder, D, indice: str = "flat", ef_search: int = 64):
        """indice: 'flat' (exacto) | 'hnsw' (FAISS IndexHNSWFlat, M=32)."""
        if indice == "hnsw":
            idx = faiss.IndexHNSWFlat(D.shape[1], 32, faiss.METRIC_INNER_PRODUCT); idx.hnsw.efConstruction = 80
        else:
            idx = faiss.IndexFlatIP(D.shape[1])
        idx.add(l2n(D))
        self.versiones[nombre] = {"emb": embedder, "idx": idx, "W": None, "base": nombre, "indice": indice, "ef": ef_search}

    def registrar_adaptador(self, nombre, embedder, W, sobre: str):
        """Queries con `embedder` (versión nueva) proyectadas con W al espacio del índice `sobre` (versión vieja)."""
        base = self.versiones[sobre]
        self.versiones[nombre] = {"emb": embedder, "idx": base["idx"], "W": W, "base": sobre, "indice": base["indice"], "ef": base["ef"]}

    def embebe_query(self, texto, nombre):
        k = clave_cache(nombre, texto)
        v = self.cache.get(k)
        if v is None:
            cfg = self.versiones[nombre]
            v = cfg["emb"].encode_queries([texto])
            if cfg["W"] is not None:
                v = l2n(v @ cfg["W"])
            self.cache.put(k, v)
        return v

    def _rangos_densos(self, texto, nombre, cand):
        return [int(i) for i in self._cands_densos(texto, nombre, cand)[0]]

    def _cands_densos(self, texto, nombre, cand):
        """(índices, scores) del top-`profundidad` denso, con el filtro aplicado DENTRO de la búsqueda."""
        cfg = self.versiones[nombre]; idx = cfg["idx"]; q = self.embebe_query(texto, nombre)
        sel = faiss.IDSelectorBatch(cand.astype("int64")) if cand is not None else None
        if cfg["indice"] == "hnsw":       # HNSW exige sus propios parámetros (efSearch + selector de filtro)
            params = faiss.SearchParametersHNSW(efSearch=cfg["ef"]) if sel is None else faiss.SearchParametersHNSW(sel=sel, efSearch=cfg["ef"])
        else:
            params = faiss.SearchParameters(sel=sel) if sel is not None else None
        Dsc, I = idx.search(q, min(self.prof, idx.ntotal), params=params)
        m = I[0] >= 0
        return I[0][m], Dsc[0][m]

    def _lineal(self, texto, nombre, cand):
        """Combinación convexa con min-max dentro de cada lista (un doc ausente de una lista aporta 0 por ella)."""
        I, sd = self._cands_densos(texto, nombre, cand)
        sb_all = self.bm25.scores(self.tok(texto))
        Ib = np.array(self._rangos_bm25(texto, cand), dtype=int); sb = sb_all[Ib] if len(Ib) else np.zeros(0)
        def mm(v):
            return (v - v.min()) / max(float(v.max() - v.min()), 1e-9) if len(v) else v
        score = {}
        for i, v in zip(I, mm(sd)):
            score[int(i)] = score.get(int(i), 0.0) + self.alfa * float(v)
        for i, v in zip(Ib, mm(sb)):
            score[int(i)] = score.get(int(i), 0.0) + (1 - self.alfa) * float(v)
        return score

    def _rangos_bm25(self, texto, cand):
        s = self.bm25.scores(self.tok(texto))
        if cand is not None:
            m = np.full(len(s), -np.inf); m[cand] = s[cand]; s = m
        orden = np.lexsort((np.arange(len(s)), -s))[: self.prof]
        return [int(i) for i in orden if np.isfinite(s[i]) and s[i] > 0]

    def buscar(self, texto, k: int = 10, lang=None, modo: str = "hibrido", version: str = "prod"):
        t0 = time.perf_counter()
        nombre = self.alias.get(version, version)
        cand = np.flatnonzero(self.lang == lang) if lang else None
        if modo == "hibrido" and self.fusion == "lineal":
            score = self._lineal(texto, nombre, cand)
        else:
            listas = []
            if modo in ("denso", "hibrido"):
                listas.append(self._rangos_densos(texto, nombre, cand))
            if modo in ("bm25", "hibrido"):
                listas.append(self._rangos_bm25(texto, cand))
            score = {}
            for lst in listas:                       # RRF: sum 1/(k + rango), rango desde 1
                for r, i in enumerate(lst):
                    score[i] = score.get(i, 0.0) + 1.0 / (self.rrf_k + r + 1)
        top = sorted(score, key=lambda i: (-score[i], i))[:k]
        self.lat_ms.append((time.perf_counter() - t0) * 1e3)
        return [(self.ids[i], round(score[i], 5)) for i in top]

    def metricas(self) -> dict:
        lat = np.array(self.lat_ms) if self.lat_ms else np.zeros(1)
        return {"n": len(self.lat_ms), "p50_ms": float(np.percentile(lat, 50)), "p99_ms": float(np.percentile(lat, 99)),
                "hit_rate": self.cache.hit_rate, "versiones": sorted(self.versiones), "alias": dict(self.alias)}

In [ ]:
def ajusta_adaptador(A, B, lam: float = 0.1, metodo: str = "ridge"):
    """Aprende W (d_src x d_dst) con A @ W ≈ B a partir de PARES (mismo texto embebido con ambas versiones).
    ridge: W = (AᵀA + λI)⁻¹ AᵀB.  procrustes: W semi-ortogonal = U Vᵀ con U Σ Vᵀ = SVD(AᵀB) (conserva ángulos)."""
    A, B = np.asarray(A, np.float64), np.asarray(B, np.float64)
    if metodo == "ridge":
        return np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ B).astype(np.float32)
    U, _, Vt = np.linalg.svd(A.T @ B, full_matrices=False)
    return (U @ Vt).astype(np.float32)

In [ ]:
def mide_latencias() -> dict:
    s = construye_servicio()
    for t in q_texts: s.buscar(t)
    for t in q_texts: s.buscar(t)
    lat = np.array(s.lat_ms); n = len(q_texts)
    z = construye_servicio(cache_cap=1024)
    for t in trafico_zipf(5000, UNIVERSO): z.buscar(t)
    return {"frio_p50": float(np.percentile(lat[:n], 50)), "frio_p99": float(np.percentile(lat[:n], 99)),
            "caliente_p50": float(np.percentile(lat[n:], 50)), "caliente_p99": float(np.percentile(lat[n:], 99)),
            "hit_rate_zipf": z.cache.hit_rate}

def evalua_adaptador(frac: float = 0.5, seed: int = SEED) -> dict:
    idx = np.random.default_rng(seed).choice(len(doc_texts), int(frac * len(doc_texts)), replace=False)
    W = ajusta_adaptador(EMB["v2"]["D"][idx], EMB["v1"]["D"][idx], lam=0.1)
    s = construye_servicio(); s.registrar_adaptador("v2→v1", VERS["v2"], W, sobre="v1")
    pq = {v: ndcg_servicio(s, modo="denso", version=n) for v, n in (("v1", "v1"), ("v2", "v2"), ("adaptador", "v2→v1"))}
    return {"v1": pq["v1"].mean(), "v2": pq["v2"].mean(), "adaptador": pq["adaptador"].mean(), "pq": pq}

def plan_migracion(pq: dict) -> dict:
    T = 10_000_000 * 400
    usd_h, tok_s, util = 0.80, 20_000, 0.6
    a, b = paired_bootstrap(pq["v2"], pq["v1"]), paired_bootstrap(pq["v2"], pq["adaptador"])
    if a["ci_low"] <= 0:
        dec = "quedarse en v1"
    elif b["ci_low"] > 0:
        dec = "re-embedding blue/green (adaptador solo como puente)"
    else:
        dec = "adaptador"
    return {"tokens_corpus_prod": T, "coste_api_usd": {k: T / 1e6 * p for k, p in PRECIOS_API.items()},
            "coste_gpu_usd": T / (tok_s * 3600 * util) * usd_h, "horas_gpu": T / (tok_s * 3600 * util),
            "supuestos": f"precios API estándar oct-2026; GPU SUPUESTA {usd_h} $/h, {tok_s} tok/s, utilización {util}; 10 M chunks x 400 tokens",
            "v2_vs_v1": a, "v2_vs_adaptador": b, "decision": dec}

In [ ]:
rubrica()
print(json.dumps({k: (v if not isinstance(v, dict) else {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items()})
                  for k, v in RES["plan"].items()}, indent=1, ensure_ascii=False))

In [ ]:
# Figura del informe: calidad por ruta de migración + latencias del servicio
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
nombres = ["v1 (prod actual)", "adaptador v2→v1 (50 %)", "v2 re-embedding"]
vals = [RES["v1"], RES["adapt50"], RES["v2"]]
cis = [bootstrap_ci(RES["_pq"][k]) for k in ("v1", "adaptador", "v2")]
axs[0].bar(nombres, vals, color=[GRIS, NARANJA, AZUL], yerr=[[v - c[1] for v, c in zip(vals, cis)], [c[2] - v for v, c in zip(vals, cis)]], capsize=4)
axs[0].set_ylabel("nDCG@10 (IC95 bootstrap)"); axs[0].set_title("Rutas de migración servidas por el servicio"); axs[0].tick_params(axis="x", labelsize=8)
axs[1].bar(["frío p50", "frío p99", "caliente p50", "caliente p99"], [RES["frio_p50"], RES["frio_p99"], RES["caliente_p50"], RES["caliente_p99"]],
           color=[NARANJA, NARANJA, AZUL, AZUL]); axs[1].set_ylabel("ms por query (CPU, batch=1)"); axs[1].set_title("Latencia del servicio")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar:** el adaptador queda entre v1 y v2 con IC que se solapan con v1: es un puente, no un destino. La latencia en caliente baja un orden de magnitud porque se salta el embedder (aquí, una llamada individual a TF-IDF+SVD de scikit-learn cuesta ~10 ms; con un embedder neuronal en una GPU remota el ahorro por acierto incluye además el viaje de red).

## 🚀 Retos extra
1. **Caché de documentos por contenido**: simula 3 releases de la documentación donde cambia el 10 % de los chunks y mide cuántos tokens re-embebes con caché `hash(texto)+versión` vs sin ella.
2. **Qdrant como backend** del servicio (named vectors `v1`/`v2` en una sola colección, alias de colección para `prod`) y compara resultados con FAISS.
3. **Boost de idioma en vez de filtro duro**: suma un término $\beta\cdot[\text{lang(doc)}=\text{lang(query)}]$ al score RRF y elige $\beta$ por validación cruzada.
4. **Micro-batching real**: un hilo que agrupe peticiones concurrentes (cola + `max_wait`) delante de `encode_queries` y mide p99 con un generador de carga Poisson.
5. **Interleaving en el servicio**: un endpoint `/search?experimento=v1_vs_v2` que devuelva la lista *team-draft* y registre a qué equipo pertenece cada resultado.

## 🤔 Reflexión (5 pts, la revisa una persona)
1. ¿Qué parte del plan cambia si el texto original de los documentos **ya no está disponible** (solo guardaste vectores)?
2. Tu servicio usa filtro **duro** de idioma. Describe una query real de Políglota donde eso empeora el resultado y cómo lo detectarías en producción.
3. ¿Qué tres métricas pondrías en el panel de la beta y con qué umbral de alerta cada una? (Conecta con §9–§10 de la lección.)

**Siguiente:** en el módulo 19 convertirás estas decisiones en árboles de decisión y casos de *system design*; en el 20, este servicio será la capa de servicio de Políglota completo.